In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mitigasi kesalahan tensor-network (TEM): Qiskit Function oleh Algorithmiq
*Lihat [referensi API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini berada dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.


<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan penggunaan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Ikhtisar
Metode Tensor-network Error Mitigation (TEM) dari Algorithmiq adalah algoritma
hybrid quantum-klasik yang dirancang untuk melakukan mitigasi noise sepenuhnya di
tahap pasca-pemrosesan klasik. Dengan TEM, pengguna dapat menghitung
nilai ekspektasi dari observable dengan memitigasi kesalahan akibat noise yang
tak terhindarkan pada perangkat keras kuantum, dengan akurasi yang lebih tinggi dan efisiensi biaya yang lebih baik,
menjadikannya pilihan yang sangat menarik bagi para peneliti kuantum maupun
praktisi industri.

Metode ini terdiri dari membangun tensor network yang merepresentasikan invers dari
kanal noise global yang memengaruhi keadaan prosesor kuantum, kemudian
menerapkan peta tersebut pada hasil pengukuran yang informationally complete yang diperoleh dari
keadaan bising untuk mendapatkan estimator yang tidak bias untuk observable.

Sebagai keunggulan, TEM memanfaatkan pengukuran yang informationally complete untuk
memberikan akses ke sejumlah besar nilai ekspektasi observable yang telah dimitigasi, dan memiliki
overhead sampling yang optimal pada perangkat keras kuantum, seperti yang dijelaskan dalam Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), dan Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542).
Overhead pengukuran mengacu pada jumlah pengukuran tambahan yang diperlukan untuk
melakukan mitigasi kesalahan yang efisien, faktor kritis dalam kelayakan
komputasi kuantum. Oleh karena itu, TEM berpotensi untuk memungkinkan
keunggulan kuantum dalam skenario yang kompleks, seperti aplikasi di bidang
quantum chaos, fisika many-body, dinamika Hubbard, dan simulasi kimia molekul kecil.

Fitur dan manfaat utama TEM dapat dirangkum sebagai berikut:

1.  **Overhead pengukuran yang optimal**: TEM optimal terhadap
[batas teoritis](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
yang berarti tidak ada metode lain yang dapat mencapai overhead pengukuran yang lebih kecil. Dengan kata lain,
TEM membutuhkan jumlah pengukuran tambahan minimum untuk melakukan
mitigasi kesalahan. Hal ini berarti TEM menggunakan runtime kuantum yang minimal.
2.  **Efisiensi biaya**: Karena TEM menangani mitigasi noise sepenuhnya di tahap
pasca-pemrosesan, tidak perlu menambahkan Circuit tambahan ke komputer kuantum,
yang tidak hanya membuat komputasi lebih murah tetapi juga mengurangi
risiko munculnya kesalahan tambahan akibat ketidaksempurnaan perangkat kuantum.
3.  **Estimasi beberapa observable**: Berkat pengukuran yang informationally complete,
TEM secara efisien mengestimasi beberapa observable dengan data pengukuran yang sama
dari komputer kuantum.
4.  **Mitigasi kesalahan pengukuran**: Qiskit Function TEM juga mencakup
[metode mitigasi kesalahan pengukuran yang dipatenkan](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
yang mampu secara signifikan mengurangi kesalahan readout setelah proses kalibrasi singkat.
5.  **Akurasi**: TEM secara signifikan meningkatkan akurasi dan keandalan

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
*See the [API reference](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*
simulasi kuantum digital, membuat algoritma kuantum lebih presisi dan
dapat diandalkan.
## Deskripsi
Fungsi TEM memungkinkan kamu mendapatkan nilai ekspektasi yang telah dimitigasi untuk
beberapa observable pada Circuit kuantum dengan overhead sampling yang minimal.
Circuit diukur dengan positive operator-valued measure (IC-POVM) yang informationally complete,
dan hasil pengukuran yang dikumpulkan diproses pada komputer klasik. Pengukuran ini digunakan
untuk menerapkan metode tensor network dan membangun peta inversi noise. Fungsi ini menerapkan
peta yang sepenuhnya membalik seluruh Circuit bising menggunakan tensor network untuk merepresentasikan
lapisan-lapisan bising.

![Skema TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimasi yang dimitigasi dari observable O melalui pasca-pemrosesan hasil pengukuran dari prosesor kuantum bising. U dan N menunjukkan operasi kuantum ideal dan peta noise terkait, yang secara umum dapat bersifat non-lokal (dan diperluas ke kotak abu-abu). D adalah tensor dari operator yang dual terhadap efek dalam pengukuran IC. Modul mitigasi noise M adalah tensor network yang dikontraksi secara efisien dari tengah ke luar. Iterasi pertama kontraksi direpresentasikan oleh garis ungu bertitik, yang kedua oleh garis putus-putus, dan yang ketiga oleh garis solid.")

Setelah Circuit dikirimkan ke fungsi, Circuit tersebut ditranspilasi dan
dioptimalkan untuk meminimalkan jumlah lapisan dengan Gate dua-Qubit (Gate yang lebih bising
pada perangkat kuantum). Noise yang memengaruhi lapisan-lapisan tersebut dipelajari melalui
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
menggunakan model noise sparse Pauli-Lindblad seperti yang dijelaskan dalam E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model noise merupakan deskripsi yang akurat tentang noise pada perangkat yang mampu
menangkap fitur-fitur halus, termasuk crosstalk antar qubit. Namun, noise pada
perangkat dapat berfluktuasi dan bergeser, sehingga noise yang dipelajari mungkin tidak akurat pada
saat estimasi dilakukan. Hal ini dapat mengakibatkan hasil yang tidak akurat.
## Mulai
Autentikasi menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/) kamu, dan pilih fungsi TEM sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akun](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokal kamu.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Contoh
Cuplikan berikut menunjukkan contoh di mana TEM digunakan untuk menghitung nilai ekspektasi dari sebuah observable pada Circuit kuantum sederhana.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Gunakan Qiskit Serverless APIs untuk memeriksa status workload Qiskit Function kamu:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.02165380888171687


Kamu bisa mendapatkan hasilnya seperti berikut: